# Portfolio Risk and Benchmark Dashboard

This notebook shows the calculations behind the Streamlit dashboard step by step.

It uses a sample portfolio, downloads market data, calculates portfolio value and risk, compares the portfolio against SPY, and creates a simple portfolio health score.


## BLOCK 0 - IMPORTS

Purpose:
Import the libraries needed for data collection, calculations, and charts.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## BLOCK 1 - CONFIGURATION

Purpose:
Store the main settings used throughout the notebook.


In [ ]:
benchmark_ticker = "SPY"
risk_free_rate = 0.02
trading_days = 252
start_date = "2023-01-01"
end_date = "2026-06-25"


## BLOCK 2 - PORTFOLIO INPUT

Purpose:
Create a sample portfolio.

In the Streamlit app, this information comes from an uploaded CSV file.


In [ ]:
portfolio = {
    "Ticker": ["AAPL", "MSFT", "NVDA"],
    "Shares": [10, 5, 3],
    "Purchase Price": [180, 420, 120]
}

portfolio_df = pd.DataFrame(portfolio)
portfolio_df


## BLOCK 3 - DATA VALIDATION

Purpose:
Check that the portfolio data is usable.

Checks:
- Required columns exist
- No blank values
- Shares are greater than zero
- Purchase prices are greater than zero


In [ ]:
required_columns = ["Ticker", "Shares", "Purchase Price"]

missing_columns = [col for col in required_columns if col not in portfolio_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

portfolio_df = portfolio_df[required_columns].copy()
portfolio_df["Ticker"] = portfolio_df["Ticker"].astype(str).str.upper().str.strip()
portfolio_df["Shares"] = pd.to_numeric(portfolio_df["Shares"], errors="coerce")
portfolio_df["Purchase Price"] = pd.to_numeric(portfolio_df["Purchase Price"], errors="coerce")

if portfolio_df.isna().any().any():
    raise ValueError("The portfolio contains blank or invalid values.")

if (portfolio_df["Shares"] <= 0).any():
    raise ValueError("Shares must be greater than zero.")

if (portfolio_df["Purchase Price"] <= 0).any():
    raise ValueError("Purchase prices must be greater than zero.")

portfolio_df


## BLOCK 4 - MARKET DATA FETCHING

Purpose:
Download historical prices for the portfolio holdings and benchmark.

Output:
- Historical prices for holdings
- Benchmark prices
- Current prices


In [ ]:
tickers = portfolio_df["Ticker"].tolist()
all_tickers = tickers + [benchmark_ticker]

market_data = yf.download(
    all_tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

historical_prices = market_data["Close"].copy()
historical_prices = historical_prices.ffill().dropna(how="all")

current_prices = historical_prices[tickers].iloc[-1]
benchmark_prices = historical_prices[benchmark_ticker].dropna()
holding_prices = historical_prices[tickers].dropna()

current_prices


## BLOCK 5 - PORTFOLIO CALCULATIONS

Purpose:
Calculate current value, cost basis, profit/loss, and return for each holding.


In [ ]:
portfolio_summary = portfolio_df.copy()
portfolio_summary["Current Price"] = portfolio_summary["Ticker"].map(current_prices)
portfolio_summary["Current Value"] = portfolio_summary["Shares"] * portfolio_summary["Current Price"]
portfolio_summary["Cost Basis"] = portfolio_summary["Shares"] * portfolio_summary["Purchase Price"]
portfolio_summary["Profit/Loss"] = portfolio_summary["Current Value"] - portfolio_summary["Cost Basis"]
portfolio_summary["Return %"] = portfolio_summary["Profit/Loss"] / portfolio_summary["Cost Basis"]

portfolio_summary


## BLOCK 6 - PORTFOLIO ALLOCATION

Purpose:
Calculate each holding's weight in the portfolio.

This helps identify concentration risk.


In [ ]:
allocation_df = portfolio_summary[["Ticker", "Current Value"]].copy()
allocation_df["Weight"] = allocation_df["Current Value"] / allocation_df["Current Value"].sum()
allocation_df = allocation_df.sort_values("Weight", ascending=False)

largest_holding = allocation_df.iloc[0]

if largest_holding["Weight"] > 0.40:
    concentration_risk = "High"
elif largest_holding["Weight"] > 0.25:
    concentration_risk = "Moderate"
else:
    concentration_risk = "Low"

allocation_df


## BLOCK 7 - RISK METRICS

Purpose:
Calculate portfolio risk and return metrics.

Metrics:
- Annual return
- Volatility
- Sharpe ratio
- Maximum drawdown


In [ ]:
daily_returns = holding_prices.pct_change().dropna()
weights = allocation_df.set_index("Ticker").loc[holding_prices.columns, "Weight"]
portfolio_daily_returns = daily_returns.dot(weights)
portfolio_growth = (1 + portfolio_daily_returns).cumprod()

annual_return = (1 + portfolio_daily_returns.mean()) ** trading_days - 1
volatility = portfolio_daily_returns.std() * np.sqrt(trading_days)
sharpe_ratio = (annual_return - risk_free_rate) / volatility
portfolio_drawdown = portfolio_growth / portfolio_growth.cummax() - 1
maximum_drawdown = portfolio_drawdown.min()

risk_summary = pd.DataFrame({
    "Metric": ["Annual Return", "Volatility", "Sharpe Ratio", "Maximum Drawdown"],
    "Value": [annual_return, volatility, sharpe_ratio, maximum_drawdown]
})

risk_summary


## BLOCK 8 - BENCHMARK COMPARISON

Purpose:
Compare portfolio performance against SPY.

This answers whether the portfolio outperformed the market benchmark and whether it took more risk.


In [ ]:
benchmark_returns = benchmark_prices.pct_change().dropna()

comparison_data = pd.concat(
    [portfolio_daily_returns.rename("Portfolio"), benchmark_returns.rename("Benchmark")],
    axis=1
).dropna()

portfolio_return = (1 + comparison_data["Portfolio"].mean()) ** trading_days - 1
benchmark_return = (1 + comparison_data["Benchmark"].mean()) ** trading_days - 1
portfolio_volatility = comparison_data["Portfolio"].std() * np.sqrt(trading_days)
benchmark_volatility = comparison_data["Benchmark"].std() * np.sqrt(trading_days)
relative_performance = portfolio_return - benchmark_return

benchmark_summary = pd.DataFrame({
    "Metric": ["Portfolio Return", "Benchmark Return", "Relative Performance", "Portfolio Volatility", "Benchmark Volatility"],
    "Value": [portfolio_return, benchmark_return, relative_performance, portfolio_volatility, benchmark_volatility]
})

benchmark_summary


## BLOCK 9 - PORTFOLIO HEALTH SCORE

Purpose:
Create a simple score that summarises diversification, risk, and return.

This score is educational and rule-based. It is not investment advice.


In [ ]:
def score_diversification(allocation_df):
    largest_weight = allocation_df["Weight"].max()
    number_of_holdings = len(allocation_df)

    if largest_weight <= 0.25:
        score = 9
    elif largest_weight <= 0.40:
        score = 7
    elif largest_weight <= 0.60:
        score = 4
    else:
        score = 2

    if number_of_holdings >= 5:
        score += 1

    return min(score, 10)


def score_risk(volatility, maximum_drawdown):
    if volatility <= 0.15:
        volatility_score = 9
    elif volatility <= 0.25:
        volatility_score = 7
    elif volatility <= 0.35:
        volatility_score = 5
    else:
        volatility_score = 3

    drawdown_size = abs(maximum_drawdown)

    if drawdown_size <= 0.15:
        drawdown_score = 9
    elif drawdown_size <= 0.25:
        drawdown_score = 7
    elif drawdown_size <= 0.40:
        drawdown_score = 5
    else:
        drawdown_score = 3

    return (volatility_score + drawdown_score) / 2


def score_return(relative_performance, sharpe_ratio, portfolio_return):
    if relative_performance > 0 and sharpe_ratio >= 1:
        return 9
    elif relative_performance > 0 or sharpe_ratio >= 0.75:
        return 7
    elif portfolio_return > 0:
        return 5
    else:
        return 3


diversification_score = score_diversification(allocation_df)
risk_score = score_risk(volatility, maximum_drawdown)
return_score = score_return(relative_performance, sharpe_ratio, portfolio_return)
overall_score = (diversification_score + risk_score + return_score) / 3

health_score = pd.DataFrame({
    "Category": ["Diversification", "Risk", "Return", "Overall Score"],
    "Score": [diversification_score, risk_score, return_score, overall_score]
})

health_score


## BLOCK 10 - VISUALIZATIONS

Purpose:
Create charts for portfolio growth, allocation, drawdown, and benchmark comparison.


In [ ]:
plt.figure(figsize=(10, 5))
portfolio_growth.plot(label="Portfolio")
plt.title("Portfolio Growth")
plt.ylabel("Growth of $1")
plt.legend()
plt.show()

plt.figure(figsize=(6, 6))
plt.pie(allocation_df["Weight"], labels=allocation_df["Ticker"], autopct="%1.1f%%", startangle=90)
plt.title("Portfolio Allocation")
plt.show()

plt.figure(figsize=(10, 5))
portfolio_drawdown.plot(color="red")
plt.title("Portfolio Drawdown")
plt.ylabel("Drawdown")
plt.show()

portfolio_vs_benchmark = (1 + comparison_data).cumprod()
portfolio_vs_benchmark.plot(figsize=(10, 5))
plt.title("Portfolio vs SPY")
plt.ylabel("Growth of $1")
plt.show()


## BLOCK 11 - CONCLUSION

Purpose:
Summarise what the project does.

This project evaluates a portfolio beyond simple profit/loss. It shows allocation, risk, benchmark comparison, and a simple health score. The health score is only a simplified guide, but it helps make several portfolio metrics easier to understand.
